In [487]:
import pandas as pd
import numpy as np
import regex as re
import locationtagger
from deep_translator import GoogleTranslator


In [488]:
product_details = pd.read_csv('product_details.csv')
product_base = pd.read_csv('product_base.csv')

/var/folders/3c/dwwhhxpn5v1_gqzr69zmy0900000gn/T/ipykernel_42211/2588855562.py:1: DtypeWarning: Columns (22,23,28,29,33,41,49,51,54,57,58,60,62,63,64,65,69,70,72,73,74,75,77,78,81,83,84,85,86,91,92,94,95,96,97,99,100,101,102,107,110,113,114,115,116,118,119,120,121,122,123,124,125,128,129,130,131,132,133,134,136,137,138,140,143,144,145,146,147,148,149,153,155,156,158,159,160,161,162,163,165,166,167,170,173,174,175,176,177,178,179,180,181,182,186,187,189,190,191,192,193,194,195,196,197,198,199,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329

In [489]:
product_base.head()

,Unnamed: 0,Referencja,EAN,Nazwa,Dane Produktowe
0,0,459,5900056015933,Kopernik - Torcik piernikowy z nadzieniem o sm...,https://apps.auchan.pl/feed/getCard/459
1,1,650,5904358550406,House of Asia - Sos Yakitori - 50 g,https://apps.auchan.pl/feed/getCard/650
2,2,667,5904358550437,House of Asia - Pasta Katsu curry - 50 g,https://apps.auchan.pl/feed/getCard/667
3,3,786,5904358552196,House of Asia - Pasta Kimchi pikantna - 50 g,https://apps.auchan.pl/feed/getCard/786
4,4,799,5904358551977,House of Asia - Papier tapiokowy cienki nie wy...,https://apps.auchan.pl/feed/getCard/799


In [490]:
product_base = product_base.drop(columns=['Unnamed: 0'])
product_base['EAN'] = product_base['EAN'].apply(str)
product_base['Referencja'] = product_base['Referencja'].apply(str)

In [491]:
product_details.shape

(11059, 666)

In [492]:
product_details.columns.tolist()

['Unnamed: 0',
 'Referencja',
 'waga',
 'zalecenia dla alergikow',
 'opis produktu',
 'marka standaryzowana',
 'jednostka opisowa',
 'kraj pochodzenia',
 'nazwa produktu ureg prawnie',
 'marka',
 'dodatki',
 'dodatkowe informacje',
 'wartość energetyczna',
 'tłuszcz',
 'tym kwasy tłuszczowe nasycone',
 'węglowodany',
 'tym cukry',
 'białko',
 'sól',
 'cechy',
 'błonnik',
 'informacje dot stylu zycia',
 'zawartość tłuszczu',
 'materia nieorganiczna',
 'włókno surowe',
 'wapń',
 'fosfor',
 'Unnamed: 27',
 'przeciwutleniacze',
 'barwniki',
 'dodatki dietetyczne',
 'witamina',
 'tauryna',
 'miedź pięciowodny siarczanu miedzi',
 'jod jodek potasu',
 'żelazo siarczan żelaza jednowodny',
 'mangan siarczan manganawy jednowodny',
 'selen selenin sodu',
 'cynk siarczan cynku jednowodny',
 'pochodzenie',
 'opakowanie zawiera porcji',
 'przeciwutleniacze barwniki',
 'miedź pięciowodny siarczan miedzi',
 'podmarka',
 'energia',
 'warunki przechowywania',
 'tym kwasy nasycone',
 'ryboflawina',
 'wit

In [493]:
product_details.dropna(axis=1, inplace=True, thresh=5)
product_details.shape

(11059, 268)

In [494]:
product_details.columns.tolist()

['Unnamed: 0',
 'Referencja',
 'waga',
 'zalecenia dla alergikow',
 'opis produktu',
 'marka standaryzowana',
 'jednostka opisowa',
 'kraj pochodzenia',
 'nazwa produktu ureg prawnie',
 'marka',
 'dodatki',
 'dodatkowe informacje',
 'wartość energetyczna',
 'tłuszcz',
 'tym kwasy tłuszczowe nasycone',
 'węglowodany',
 'tym cukry',
 'białko',
 'sól',
 'cechy',
 'błonnik',
 'informacje dot stylu zycia',
 'zawartość tłuszczu',
 'materia nieorganiczna',
 'włókno surowe',
 'wapń',
 'fosfor',
 'Unnamed: 27',
 'przeciwutleniacze',
 'barwniki',
 'dodatki dietetyczne',
 'witamina',
 'tauryna',
 'miedź pięciowodny siarczanu miedzi',
 'jod jodek potasu',
 'żelazo siarczan żelaza jednowodny',
 'mangan siarczan manganawy jednowodny',
 'selen selenin sodu',
 'cynk siarczan cynku jednowodny',
 'pochodzenie',
 'opakowanie zawiera porcji',
 'przeciwutleniacze barwniki',
 'miedź pięciowodny siarczan miedzi',
 'podmarka',
 'energia',
 'warunki przechowywania',
 'tym kwasy nasycone',
 'ryboflawina',
 'wit

In [495]:
product_details[product_details['tłuszcz surowy'].notna()]

,Unnamed: 0,Referencja,waga,zalecenia dla alergikow,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,nazwa produktu ureg prawnie,marka,...,fluorek,cholina,opakowanie zawiera porcję produktu przygotowaniu,tym skrobia,przeciwutleniacze konserwanty,butelka zawiera porcji,nrv referencyjne wartości spożycia,aniony,dzienne referencyjne wartości spożycia witamin,porcja gałka gałka
45,45,15098,NaN,NaN,Butcher's z wołowiną to pełnoporcjowa karma d...,Butcher's,kg,Kraj pochodzenia - Wielka Brytania,NaN,Butcher's,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,125,30106,NaN,NaN,Pełnoporcjowa karma dla dorosłych kotów. Wido...,Purina ONE,kg,NaN,NaN,PURINA ONE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194,194,13141,NaN,NaN,Butcher's z kurczakiem to pełnoporcjowa karma...,Butcher's,kg,Kraj pochodzenia - Wielka Brytania,NaN,Butcher's,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
304,304,80404,NaN,NaN,Pełnoporcjowa karma dla dorosłych psów.,Friskies,kg,NaN,NaN,riskies,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
361,361,122536,NaN,NaN,Darling Dog z pyszną mieszanką Kurczaka i Ind...,Darling,kg,NaN,NaN,Darling,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10546,10546,846456,NaN,NaN,Pełnoporcjowa karma dla kociąt. Odpowiednia r...,Felix,g,NaN,NaN,Felix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10632,10632,971692,NaN,NaN,Pełnoporcjowa karma dla dorosłych kotów. Wido...,Purina ONE,g,NaN,NaN,PURINA ONE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10649,10649,998809,Waga brutto - 190 g,NaN,Uzupełniająca karma dla dorosłych psów. Frisk...,Friskies,NaN,NaN,NaN,Friskies,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10817,10817,883010,NaN,NaN,Pełnoporcjowa karma dla dorosłych kotów. Karm...,Felix,g,NaN,NaN,Felix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [496]:
mapping = {
    "zawartość tłuszczu": "tłuszcz",
    "tłuszcz tym": "tłuszcz",
    "tluszcz": "tłuszcz",
    "tłuszcze": "tłuszcz",
    "tuszcz": "tłuszcz",

    "bialko": "białko",
    "białka": "białko",

    "węglowodany tym": "węglowodany",
    "weglowodany": "węglowodany",

    "tym cukry": "cukry",
    "cukry ogółem": "cukry",
    "cukier": "cukry",

    "energia": "wartość energetyczna",
    "wartość energetyczna kcal": "wartość energetyczna",
    "wartość energetyczna energia kcal": "wartość energetyczna",
    "wartosc energetyczna": "wartość energetyczna",
    "wartosc energetyczna kcal": "wartość energetyczna",
    "blonnik": "wartość energetyczna",
    "wartość energetyczna energia": "wartość energetyczna",
    "energia kcal": "wartość energetyczna",

    "sol": "sól",
}

In [497]:
# Standardizing column names 

for source_col, target_col in mapping.items():
    if source_col in product_details.columns:
        product_details[target_col] = product_details[target_col].combine_first(product_details[source_col])

cols_to_drop = [col for col in product_details.columns if col in mapping]
product_details = product_details.drop(columns=cols_to_drop)

In [498]:
print(product_details.isna().sum().tolist())

[0, 0, 6996, 6412, 5052, 6258, 5824, 5819, 2771, 1755, 9233, 9575, 5029, 4841, 6024, 4907, 4737, 5050, 7808, 8644, 9720, 10957, 10797, 10778, 10956, 10692, 11010, 11048, 10840, 10489, 10934, 11041, 11024, 10993, 10978, 11035, 10978, 9988, 10954, 11044, 10995, 9460, 9230, 10337, 10990, 10970, 9448, 11052, 10454, 10211, 11053, 10988, 10968, 11026, 11036, 11008, 11040, 11005, 11054, 11042, 11054, 11034, 10941, 10905, 10729, 10977, 11053, 10993, 10991, 10953, 11045, 11029, 11047, 11048, 11052, 10123, 10983, 11046, 11046, 10958, 11046, 11045, 11050, 11043, 10883, 11046, 11051, 10938, 10973, 10994, 10992, 11047, 4945, 11045, 10995, 10924, 11049, 10977, 10950, 11036, 10995, 11054, 11050, 11022, 11050, 10981, 11025, 11033, 11000, 10924, 11029, 11051, 11047, 11037, 11003, 11009, 11038, 11022, 11048, 11048, 11037, 11053, 11052, 11048, 11048, 11051, 11053, 11036, 11036, 11050, 11054, 11050, 11053, 11022, 11016, 11017, 11049, 11005, 11053, 11028, 11052, 11051, 11051, 11049, 11051, 11046, 11053, 11

In [499]:
product_details.shape

(11059, 247)

In [500]:
cols_to_drop = [col for col in product_details.columns if product_details[col].isna().sum() > 10000]

product_details = product_details.drop(columns=cols_to_drop)

In [501]:
product_details.shape

(11059, 26)

In [502]:
product_details.head()

,Unnamed: 0,Referencja,waga,zalecenia dla alergikow,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,nazwa produktu ureg prawnie,marka,...,białko,sól,cechy,błonnik,informacje dot stylu zycia,pochodzenie,podmarka,warunki przechowywania,uzytkowanie i przechowywanie,cukry
0,0,459,Waga brutto - 212 g,"MIGDAŁY - Może zawierać, ORZECHY BRAZYLIJSKIE...",Torcik piernikowy przekładany nadzieniem o sm...,Kopernik,g,Kraj pochodzenia - Polska,Piernik w czekoladzie przekładany nadzieniem ...,Kopernik,...,"4,5 g","0,20 g",NaN,NaN,NaN,NaN,NaN,NaN,NaN,37 g
1,1,650,Waga brutto - 58 g,"SELER - Zawiera, RYBA - Zawiera, SOJA - Zawie...","Sos w japońskim stylu, o słodko - pikantnym s...",House of Asia,g,Tajlandia,Sos do Yakitori. Produkt sterylizowany.,House of Asia,...,"1,6 g","5,1 g","kuchnia japońska, słodki grill\r\n","0,2 g",NaN,NaN,NaN,NaN,NaN,27 g
2,2,667,Waga brutto - 58 g,NaN,Aromatyczna pasta z unikatową kompozycją przy...,House of Asia,g,Kraj pochodzenia - Tajlandia,Pasta Katsu curry. Produkt pasteryzowany.,House of Asia,...,"3,1 g","5,7 g","kuchnia japońska, aromatyczna\r\n","3,9 g",NaN,NaN,NaN,NaN,NaN,"3,7 g"
3,3,786,Waga brutto - 60 g,"RYBA - Zawiera, SOJA - Zawiera","Mieszanka chili, miso, czosnku, imbiru i inny...",House of Asia,g,Kraj pochodzenia - Tajlandia,Pasta do Kimchi.,House of Asia,...,"2,8 g","2,6 g",kuchnia koreańska\r\n,"5,4 g",NaN,NaN,NaN,NaN,NaN,15 g
4,4,799,Waga brutto - 62 g,NaN,NaN,House of Asia,g,Kraj pochodzenia - Wietnam,Papier tapiokowy cienki - nie wymaga namaczan...,House of Asia,...,0 g,"1,12 g","bez namaczania, naturalnie bezglutenowyDodatk...",NaN,Bezglutenowy,NaN,NaN,NaN,NaN,0 g


In [503]:
print(product_details.isna().sum())

Unnamed: 0                          0
Referencja                          0
waga                             6996
zalecenia dla alergikow          6412
opis produktu                    5052
marka standaryzowana             6258
jednostka opisowa                5824
kraj pochodzenia                 5819
nazwa produktu ureg prawnie      2771
marka                            1755
dodatki                          9233
dodatkowe informacje             9575
wartość energetyczna             5029
tłuszcz                          4841
tym kwasy tłuszczowe nasycone    6024
węglowodany                      4907
białko                           4737
sól                              5050
cechy                            7808
błonnik                          8644
informacje dot stylu zycia       9720
pochodzenie                      9988
podmarka                         9460
warunki przechowywania           9230
uzytkowanie i przechowywanie     9448
cukry                            4945
dtype: int64

In [504]:
cols_to_drop = ['Unnamed: 0',
                'zalecenia dla alergikow',
                'uzytkowanie i przechowywanie',
                'informacje dot stylu zycia', 
                'dodatkowe informacje',
                'dodatki',
                'uzytkowanie i przechowywanie',
                'tym kwasy tłuszczowe nasycone',
                'warunki przechowywania',
                'cechy']
product_details = product_details.drop(columns=cols_to_drop)

In [505]:
product_details['Referencja'] = product_details['Referencja'].apply(str)

In [506]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,nazwa produktu ureg prawnie,marka,wartość energetyczna,tłuszcz,węglowodany,białko,sól,błonnik,pochodzenie,podmarka,cukry
0,459,Waga brutto - 212 g,Torcik piernikowy przekładany nadzieniem o sm...,Kopernik,g,Kraj pochodzenia - Polska,Piernik w czekoladzie przekładany nadzieniem ...,Kopernik,1534 kJ / 364 kcal,"7,6 g",67 g,"4,5 g","0,20 g",NaN,NaN,NaN,37 g
1,650,Waga brutto - 58 g,"Sos w japońskim stylu, o słodko - pikantnym s...",House of Asia,g,Tajlandia,Sos do Yakitori. Produkt sterylizowany.,House of Asia,709 kJ / 167 kcal,0 g,40 g,"1,6 g","5,1 g","0,2 g",NaN,NaN,27 g
2,667,Waga brutto - 58 g,Aromatyczna pasta z unikatową kompozycją przy...,House of Asia,g,Kraj pochodzenia - Tajlandia,Pasta Katsu curry. Produkt pasteryzowany.,House of Asia,1289 kJ/313 kcal,30 g,"5,6 g","3,1 g","5,7 g","3,9 g",NaN,NaN,"3,7 g"
3,786,Waga brutto - 60 g,"Mieszanka chili, miso, czosnku, imbiru i inny...",House of Asia,g,Kraj pochodzenia - Tajlandia,Pasta do Kimchi.,House of Asia,668 kJ / 160 kcal,"5,5 g",22 g,"2,8 g","2,6 g","5,4 g",NaN,NaN,15 g
4,799,Waga brutto - 62 g,NaN,House of Asia,g,Kraj pochodzenia - Wietnam,Papier tapiokowy cienki - nie wymaga namaczan...,House of Asia,1442 kJ / 340 kcal,0 g,84 g,0 g,"1,12 g",NaN,NaN,NaN,0 g


In [507]:
product_details['waga'] = product_details['waga'].apply(lambda x: re.search(r"\d+\.?\d*", x).group() if isinstance(x, str) and re.search(r"\d+\.?\d*", x) else None)
product_details['waga'] = product_details['waga'].apply(pd.to_numeric)

In [508]:
product_details['kraj pochodzenia'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 11059 entries, 0 to 11058
Series name: kraj pochodzenia
Non-Null Count  Dtype 
--------------  ----- 
5240 non-null   object
dtypes: object(1)
memory usage: 86.5+ KB


In [509]:
# translator = GoogleTranslator(source='auto', target='en')

In [510]:
# product_details['kraj pochodzenia EN'] = product_details['kraj pochodzenia'].apply(lambda x: translator.translate(x) if isinstance(x, str) else None)

In [511]:
# product_details['kraj pochodzenia EN']

In [512]:
# import nltk
# nltk.downloader.download('maxent_ne_chunker')
# nltk.downloader.download('words')
# nltk.downloader.download('treebank')
# nltk.downloader.download('maxent_treebank_pos_tagger')
# nltk.downloader.download('punkt')
# nltk.download('averaged_perceptron_tagger_eng')
# nltk.download('maxent_ne_chunker_tab')

In [513]:
# import geograpy
# import nltk
# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('maxent_ne_chunker')
# nltk.download('words')

In [514]:
# product_details['kraj'] = product_details['kraj pochodzenia EN'].apply(lambda x: geograpy.get_geoPlace_context(text = x).countries if isinstance(x, str) else None)

# # product_details['kraj pochodzenia'] = product_details['kraj pochodzenia'].apply(lambda x: x.strip().lower().replace('kraj pochodzenia', '') if isinstance(x, str) else None)
# # product_details['kraj pochodzenia'] = product_details['kraj pochodzenia'].apply(lambda x: x.strip().lower().replace('wyprodukowano w ', '') if isinstance(x, str) else None)

In [515]:
# product_details['kraj'].value_counts()

In [516]:
product_details = product_details.drop(columns=['nazwa produktu ureg prawnie', 'pochodzenie', 'podmarka'])

In [517]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,białko,sól,błonnik,cukry
0,459,212.0,Torcik piernikowy przekładany nadzieniem o sm...,Kopernik,g,Kraj pochodzenia - Polska,Kopernik,1534 kJ / 364 kcal,"7,6 g",67 g,"4,5 g","0,20 g",NaN,37 g
1,650,58.0,"Sos w japońskim stylu, o słodko - pikantnym s...",House of Asia,g,Tajlandia,House of Asia,709 kJ / 167 kcal,0 g,40 g,"1,6 g","5,1 g","0,2 g",27 g
2,667,58.0,Aromatyczna pasta z unikatową kompozycją przy...,House of Asia,g,Kraj pochodzenia - Tajlandia,House of Asia,1289 kJ/313 kcal,30 g,"5,6 g","3,1 g","5,7 g","3,9 g","3,7 g"
3,786,60.0,"Mieszanka chili, miso, czosnku, imbiru i inny...",House of Asia,g,Kraj pochodzenia - Tajlandia,House of Asia,668 kJ / 160 kcal,"5,5 g",22 g,"2,8 g","2,6 g","5,4 g",15 g
4,799,62.0,NaN,House of Asia,g,Kraj pochodzenia - Wietnam,House of Asia,1442 kJ / 340 kcal,0 g,84 g,0 g,"1,12 g",NaN,0 g


In [518]:
product_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11059 entries, 0 to 11058
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Referencja            11059 non-null  object 
 1   waga                  4028 non-null   float64
 2   opis produktu         6007 non-null   object 
 3   marka standaryzowana  4801 non-null   object 
 4   jednostka opisowa     5235 non-null   object 
 5   kraj pochodzenia      5240 non-null   object 
 6   marka                 9304 non-null   object 
 7   wartość energetyczna  6030 non-null   object 
 8   tłuszcz               6218 non-null   object 
 9   węglowodany           6152 non-null   object 
 10  białko                6322 non-null   object 
 11  sól                   6009 non-null   object 
 12  błonnik               2415 non-null   object 
 13  cukry                 6114 non-null   object 
dtypes: float64(1), object(13)
memory usage: 1.2+ MB


In [519]:
cols_to_standardize = ['tłuszcz', 'węglowodany', 'białko', 'sól', 'błonnik', 'cukry']

In [520]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,białko,sól,błonnik,cukry
0,459,212.0,Torcik piernikowy przekładany nadzieniem o sm...,Kopernik,g,Kraj pochodzenia - Polska,Kopernik,1534 kJ / 364 kcal,"7,6 g",67 g,"4,5 g","0,20 g",NaN,37 g
1,650,58.0,"Sos w japońskim stylu, o słodko - pikantnym s...",House of Asia,g,Tajlandia,House of Asia,709 kJ / 167 kcal,0 g,40 g,"1,6 g","5,1 g","0,2 g",27 g
2,667,58.0,Aromatyczna pasta z unikatową kompozycją przy...,House of Asia,g,Kraj pochodzenia - Tajlandia,House of Asia,1289 kJ/313 kcal,30 g,"5,6 g","3,1 g","5,7 g","3,9 g","3,7 g"
3,786,60.0,"Mieszanka chili, miso, czosnku, imbiru i inny...",House of Asia,g,Kraj pochodzenia - Tajlandia,House of Asia,668 kJ / 160 kcal,"5,5 g",22 g,"2,8 g","2,6 g","5,4 g",15 g
4,799,62.0,NaN,House of Asia,g,Kraj pochodzenia - Wietnam,House of Asia,1442 kJ / 340 kcal,0 g,84 g,0 g,"1,12 g",NaN,0 g


In [521]:
for col in cols_to_standardize:
    product_details[col] = product_details[col].apply(lambda x: x.lower().replace("g", "").replace(",", ".").replace("<0.5", "0").replace("<0.1", "0").replace("-", "0").strip() if isinstance(x, str) else None)

In [522]:
for col in cols_to_standardize:
    product_details[col] = (
    product_details[col]
      .astype(str)
      .str.replace(',', '.', regex=False)
      .str.extract(r'([-+]?\d*\.?\d+)')[0]
      .astype(float)
      .round(2)
)

In [523]:
product_details['wartość energetyczna'].to_list()

['1534 kJ / 364 kcal',
 '709 kJ / 167 kcal',
 '1289 kJ/313 kcal',
 '668 kJ / 160 kcal',
 '1442 kJ / 340 kcal',
 '379 kcal/100 g',
 nan,
 nan,
 nan,
 '97 kJ / 23 kcal',
 nan,
 '2 kJ / 0 kcal',
 '416 kcal / 100 g',
 nan,
 nan,
 '1171 kJ / 276 kcal',
 nan,
 '365 kcal',
 '325 kJ/78 kcal',
 '155 kJ - 37 kcal',
 nan,
 ' 191 kJ / 46 kcal',
 '342 kcal',
 nan,
 '295 kJ / 70 kcal',
 '318 kJ / 75 kcal',
 nan,
 nan,
 '1506 kJ/ 363 kcal',
 nan,
 nan,
 '348,6 kJ / 83,7 kcal',
 '751 kJ / 180 kcal',
 '597 kJ/145 kcal',
 '406 kJ/97 kcal',
 '1173 kJ/282 kcal',
 '2193 kJ / 525 kcal',
 '112 kJ / 26 kcal',
 '391 kJ / 93 kcal',
 nan,
 nan,
 nan,
 nan,
 '1278',
 ' 1898 kJ / 451 kcal',
 nan,
 '414 kcal',
 nan,
 '397kcal',
 '106 kJ / 254 kcal',
 '1416 kJ / 333 kcal',
 '763 kJ / 184 kcal',
 '1395 kJ/ 332 kcal',
 '1723 kJ / 409 kcal',
 '35 kJ / 8 kcal',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 '459 kcal',
 '2102 kJ / 509 kcal',
 '584 kcal',
 '538 kcal',
 '182 kJ/43 kcal',
 '350 kcal',
 '556 kJ/134 kcal',
 nan,
 nan,

In [524]:
product_details['wartość energetyczna'] = (
    product_details['wartość energetyczna']
        .astype(str)
        .str.split('/')
        .str[-1]
        .str.replace(',', '.', regex=False)
        .str.lower()
        .str.strip()
)

In [525]:
mask = (
    product_details['wartość energetyczna'].str.contains("kj", case=False, na=False)
    & ~product_details['wartość energetyczna'].str.contains("kcal", case=False, na=False)
)


# extract kJ as float
kj = (
    product_details.loc[mask, 'wartość energetyczna']
        .str.extract(r'(\d*\.?\d+)')[0]
        .astype(float)
)

# convert to kcal
kcal = (kj / 4.184).round(2)

# write back (numeric kcal)
product_details.loc[mask, 'wartość energetyczna'] = kcal

In [526]:
product_details.isna().sum()

Referencja                 0
waga                    7031
opis produktu           5052
marka standaryzowana    6258
jednostka opisowa       5824
kraj pochodzenia        5819
marka                   1755
wartość energetyczna       0
tłuszcz                 4903
węglowodany             4962
białko                  4786
sól                     5109
błonnik                 8715
cukry                   5000
dtype: int64

In [527]:
mask = (
    product_details['wartość energetyczna'].isna()
    & product_details[['tłuszcz', 'węglowodany', 'białko']].notna().all(axis=1)
)

product_details.loc[mask, 'wartość energetyczna'] = (
    product_details.loc[mask, 'tłuszcz'] * 9
    + product_details.loc[mask, 'węglowodany'] * 4
    + product_details.loc[mask, 'białko'] * 4
)

In [528]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,białko,sól,błonnik,cukry
0,459,212.0,Torcik piernikowy przekładany nadzieniem o sm...,Kopernik,g,Kraj pochodzenia - Polska,Kopernik,364 kcal,7.6,67.0,4.5,0.20,NaN,37.0
1,650,58.0,"Sos w japońskim stylu, o słodko - pikantnym s...",House of Asia,g,Tajlandia,House of Asia,167 kcal,0.0,40.0,1.6,5.10,0.2,27.0
2,667,58.0,Aromatyczna pasta z unikatową kompozycją przy...,House of Asia,g,Kraj pochodzenia - Tajlandia,House of Asia,313 kcal,30.0,5.6,3.1,5.70,3.9,3.7
3,786,60.0,"Mieszanka chili, miso, czosnku, imbiru i inny...",House of Asia,g,Kraj pochodzenia - Tajlandia,House of Asia,160 kcal,5.5,22.0,2.8,2.60,5.4,15.0
4,799,62.0,NaN,House of Asia,g,Kraj pochodzenia - Wietnam,House of Asia,340 kcal,0.0,84.0,0.0,1.12,NaN,0.0


In [529]:
def extract_kcal(x):
    if isinstance(x, (int, float)) and not pd.isna(x):
        return float(x)
    if isinstance(x, str):
        match = re.search(r'(\d*\.?\d+)\s*kcal', x.lower())
        if match:
            return float(match.group(1))
        return np.nan

    return np.nan


product_details['wartość energetyczna'] = (
    product_details['wartość energetyczna']
        .apply(extract_kcal)
)

In [530]:
product_details['wartość energetyczna'].to_list()[:20]

[364.0,
 167.0,
 313.0,
 160.0,
 340.0,
 nan,
 nan,
 nan,
 nan,
 23.0,
 nan,
 0.0,
 nan,
 nan,
 nan,
 276.0,
 nan,
 365.0,
 78.0,
 37.0]

In [531]:
product_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11059 entries, 0 to 11058
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Referencja       11059 non-null  object
 1   EAN              11059 non-null  object
 2   Nazwa            11059 non-null  object
 3   Dane Produktowe  11059 non-null  object
dtypes: object(4)
memory usage: 345.7+ KB


In [532]:
product_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11059 entries, 0 to 11058
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Referencja            11059 non-null  object 
 1   waga                  4028 non-null   float64
 2   opis produktu         6007 non-null   object 
 3   marka standaryzowana  4801 non-null   object 
 4   jednostka opisowa     5235 non-null   object 
 5   kraj pochodzenia      5240 non-null   object 
 6   marka                 9304 non-null   object 
 7   wartość energetyczna  5683 non-null   float64
 8   tłuszcz               6156 non-null   float64
 9   węglowodany           6097 non-null   float64
 10  białko                6273 non-null   float64
 11  sól                   5950 non-null   float64
 12  błonnik               2344 non-null   float64
 13  cukry                 6059 non-null   float64
dtypes: float64(8), object(6)
memory usage: 1.2+ MB


In [533]:
product_details[product_details['Referencja'] == '43444']

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,białko,sól,błonnik,cukry
8967,43444,NaN,Cebula bogata jest w substancje bakteriobójcz...,NaN,NaN,NaN,Cebula,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [534]:
product_details['kategoria'] = np.where(product_details['wartość energetyczna'].notna() | product_base['Nazwa'].str.contains(r'warzywa auchan|owoce auchan', case=False, na=False), 'Food', 'Not food')

In [535]:
product_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11059 entries, 0 to 11058
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Referencja            11059 non-null  object 
 1   waga                  4028 non-null   float64
 2   opis produktu         6007 non-null   object 
 3   marka standaryzowana  4801 non-null   object 
 4   jednostka opisowa     5235 non-null   object 
 5   kraj pochodzenia      5240 non-null   object 
 6   marka                 9304 non-null   object 
 7   wartość energetyczna  5683 non-null   float64
 8   tłuszcz               6156 non-null   float64
 9   węglowodany           6097 non-null   float64
 10  białko                6273 non-null   float64
 11  sól                   5950 non-null   float64
 12  błonnik               2344 non-null   float64
 13  cukry                 6059 non-null   float64
 14  kategoria             11059 non-null  object 
dtypes: float64(8), obje

In [536]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,białko,sól,błonnik,cukry,kategoria
0,459,212.0,Torcik piernikowy przekładany nadzieniem o sm...,Kopernik,g,Kraj pochodzenia - Polska,Kopernik,364.0,7.6,67.0,4.5,0.20,NaN,37.0,Food
1,650,58.0,"Sos w japońskim stylu, o słodko - pikantnym s...",House of Asia,g,Tajlandia,House of Asia,167.0,0.0,40.0,1.6,5.10,0.2,27.0,Food
2,667,58.0,Aromatyczna pasta z unikatową kompozycją przy...,House of Asia,g,Kraj pochodzenia - Tajlandia,House of Asia,313.0,30.0,5.6,3.1,5.70,3.9,3.7,Food
3,786,60.0,"Mieszanka chili, miso, czosnku, imbiru i inny...",House of Asia,g,Kraj pochodzenia - Tajlandia,House of Asia,160.0,5.5,22.0,2.8,2.60,5.4,15.0,Food
4,799,62.0,NaN,House of Asia,g,Kraj pochodzenia - Wietnam,House of Asia,340.0,0.0,84.0,0.0,1.12,NaN,0.0,Food


In [537]:
product_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11059 entries, 0 to 11058
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Referencja            11059 non-null  object 
 1   waga                  4028 non-null   float64
 2   opis produktu         6007 non-null   object 
 3   marka standaryzowana  4801 non-null   object 
 4   jednostka opisowa     5235 non-null   object 
 5   kraj pochodzenia      5240 non-null   object 
 6   marka                 9304 non-null   object 
 7   wartość energetyczna  5683 non-null   float64
 8   tłuszcz               6156 non-null   float64
 9   węglowodany           6097 non-null   float64
 10  białko                6273 non-null   float64
 11  sól                   5950 non-null   float64
 12  błonnik               2344 non-null   float64
 13  cukry                 6059 non-null   float64
 14  kategoria             11059 non-null  object 
dtypes: float64(8), obje

In [538]:
import unicodedata

def ensure_str(x):
    if isinstance(x, bytes):
        return x.decode('utf-8', errors='replace')
    return str(x)

def clean_text(x):
    x = ensure_str(x)
    x = x.strip()
    x = unicodedata.normalize('NFC', x)
    x = x.replace("-", "")
    return x


In [539]:
product_base['Nazwa'] = product_base['Nazwa'].apply(ensure_str)
product_base['Nazwa'] = product_base['Nazwa'].apply(clean_text)

In [540]:
product_details['opis produktu'] = product_details['opis produktu'].apply(ensure_str)
product_details['marka standaryzowana'] = product_details['marka standaryzowana'].apply(ensure_str)
product_details['marka'] = product_details['marka'].apply(ensure_str)
product_details['opis produktu'] = product_details['opis produktu'].apply(clean_text)
product_details['marka standaryzowana'] = product_details['marka standaryzowana'].apply(clean_text)
product_details['marka'] = product_details['marka'].apply(clean_text)

In [541]:
def safe_float(x):
    """Convert to float, return None if invalid"""
    try:
        x = str(x).replace(",", ".").strip()
        x = ''.join(c for c in x if c.isdigit() or c == '.')
        return float(x) if x else None
    except (ValueError, TypeError):
        return None

def clean_text(x):
    """Strip, normalize, and return None if NaN"""
    if pd.isna(x):
        return None
    x = str(x).strip()
    return unicodedata.normalize("NFC", x)


In [542]:
string_columns = ["marka", "jednostka opisowa", "kategoria", "Referencja", "Nazwa", "Dane Produktowe", "EAN"]

for col in string_columns:
    if col in product_details.columns:
        product_details[col] = product_details[col].apply(clean_text)
    if col in product_base.columns:
        product_base[col] = product_base[col].apply(clean_text)

In [543]:
numeric_columns = ["waga", "wartość energetyczna", "tłuszcz", "węglowodany", "cukry", "białko", "sól"]

for col in numeric_columns:
    if col in product_details.columns:
        product_details[col] = product_details[col].apply(safe_float)

In [544]:
product_details = product_details.fillna("")

In [545]:
product_details.to_csv('product_details_cleaned.csv')
product_base.to_csv('product_base_cleaned.csv')

product_details.to_json('product_details_cleaned.json')
product_base.to_json('product_base_cleaned.json')